# Retail Analytics Dashboard

This notebook analyzes retail sales data including:
- Revenue trends by category and customer segment
- Top customers and products
- Order status distribution
- Profit margin analysis

In [ ]:
# Connect to Snowflake
from snowflake.snowpark.context import get_active_session
import pandas as pd
import altair as alt

session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

## Revenue by Category

In [ ]:
revenue_by_category = session.sql("""
    SELECT 
        p.category,
        SUM(oi.quantity * oi.unit_price) as revenue,
        SUM(oi.quantity) as units_sold,
        COUNT(DISTINCT o.order_id) as orders
    FROM JACK.DEMO.ORDER_ITEMS oi
    JOIN JACK.DEMO.ORDERS o ON oi.order_id = o.order_id
    JOIN JACK.DEMO.PRODUCTS p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""").to_pandas()

chart = alt.Chart(revenue_by_category).mark_bar().encode(
    x=alt.X('CATEGORY:N', sort='-y', title='Category'),
    y=alt.Y('REVENUE:Q', title='Revenue ($)'),
    color=alt.Color('CATEGORY:N', legend=None)
).properties(title='Revenue by Product Category', width=500, height=300)
chart

## Top 10 Customers by Revenue

In [ ]:
top_customers = session.sql("""
    SELECT 
        c.first_name || ' ' || c.last_name as customer_name,
        c.customer_segment,
        c.state,
        SUM(oi.quantity * oi.unit_price) as total_revenue,
        COUNT(DISTINCT o.order_id) as order_count
    FROM JACK.DEMO.CUSTOMERS c
    JOIN JACK.DEMO.ORDERS o ON c.customer_id = o.customer_id
    JOIN JACK.DEMO.ORDER_ITEMS oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.customer_segment, c.state
    ORDER BY total_revenue DESC
    LIMIT 10
""").to_pandas()

chart = alt.Chart(top_customers).mark_bar().encode(
    y=alt.Y('CUSTOMER_NAME:N', sort='-x', title='Customer'),
    x=alt.X('TOTAL_REVENUE:Q', title='Total Revenue ($)'),
    color=alt.Color('CUSTOMER_SEGMENT:N', title='Segment')
).properties(title='Top 10 Customers by Revenue', width=500, height=300)
chart

## Monthly Revenue Trend

In [ ]:
monthly_trend = session.sql("""
    SELECT 
        DATE_TRUNC('month', o.order_date) as month,
        SUM(oi.quantity * oi.unit_price) as revenue,
        COUNT(DISTINCT o.order_id) as orders
    FROM JACK.DEMO.ORDERS o
    JOIN JACK.DEMO.ORDER_ITEMS oi ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
    ORDER BY month
""").to_pandas()

chart = alt.Chart(monthly_trend).mark_line(point=True).encode(
    x=alt.X('MONTH:T', title='Month'),
    y=alt.Y('REVENUE:Q', title='Revenue ($)')
).properties(title='Monthly Revenue Trend', width=600, height=300)
chart

## Profit Margin Analysis (from Dynamic Table)

In [ ]:
profit_analysis = session.sql("""
    SELECT 
        category,
        SUM(revenue) as total_revenue,
        SUM(profit) as total_profit,
        ROUND(100 * SUM(profit) / NULLIF(SUM(revenue), 0), 2) as profit_margin_pct
    FROM JACK.DEMO.REVENUE_METRICS
    GROUP BY category
    ORDER BY profit_margin_pct DESC
""").to_pandas()

chart = alt.Chart(profit_analysis).mark_bar().encode(
    x=alt.X('CATEGORY:N', sort='-y', title='Category'),
    y=alt.Y('PROFIT_MARGIN_PCT:Q', title='Profit Margin (%)')
).properties(title='Profit Margin by Category', width=500, height=300)
chart

## Customer Segment Comparison

In [ ]:
segment_analysis = session.sql("""
    SELECT 
        customer_segment,
        SUM(revenue) as total_revenue,
        SUM(order_count) as total_orders,
        ROUND(SUM(revenue) / NULLIF(SUM(order_count), 0), 2) as avg_order_value
    FROM JACK.DEMO.REVENUE_METRICS
    GROUP BY customer_segment
""").to_pandas()

segment_analysis